# Phase 4~5 재현 실행: 학습, 평가, 점수, 후보 선정

이 노트북은 테스트된 프로젝트 서비스를 호출하는 얇은 orchestration 계층입니다. 데이터 전처리나 모델 로직을 cell에 복제하지 않습니다. 동일 export manifest, versioned YAML, seed를 사용하면 동일 run identity를 재생성합니다.

In [1]:
from __future__ import annotations

import importlib.metadata
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('pyproject.toml이 있는 저장소 루트를 찾지 못했습니다.')

REPO_ROOT = find_repo_root(Path.cwd())
for import_path in (REPO_ROOT, REPO_ROOT / 'src'):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

versions = {name: importlib.metadata.version(name) for name in ['pyqlib', 'lightgbm', 'scikit-learn', 'pandas', 'numpy']}
assert sys.version_info[:2] == (3, 11), sys.version
display(pd.Series(versions, name='version').to_frame())
REPO_ROOT


,version
pyqlib,0.9.7
lightgbm,4.7.0
scikit-learn,1.9.1
pandas,3.0.6
numpy,2.4.6


WindowsPath('C:/Users/annajeong/repos/sp500-candidate-engine')

## 실행 입력 고정

아래 run ID, config hash, export content hash가 재현성 identity를 구성합니다.

In [2]:
from scripts.run_phase4_5 import discover_export
from sp500_candidate_engine.analytics.config import load_versioned_config

EXPORT_RUN_ID = '351a5af0-5bc1-50d0-ab20-fa8292afe787'
EXPORT_DIR = discover_export(REPO_ROOT, EXPORT_RUN_ID)
MODEL_CONFIG = load_versioned_config(REPO_ROOT / 'config/analytics/model_lightgbm_v1.yaml', required_keys=('seed', 'model', 'walk_forward', 'baselines'))
SELECTION_CONFIG = load_versioned_config(REPO_ROOT / 'config/analytics/selection_v1.yaml', required_keys=('minimum_candidates', 'maximum_candidates', 'sector_cap', 'relaxed_sector_cap'))
export_manifest = json.loads((EXPORT_DIR / 'manifest.json').read_text(encoding='utf-8'))
display(pd.Series({
    'export_run_id': export_manifest['export_run_id'],
    'export_content_hash': export_manifest['content_hash'],
    'model_config_hash': MODEL_CONFIG.config_hash,
    'selection_config_hash': SELECTION_CONFIG.config_hash,
    'seed': MODEL_CONFIG.payload['seed'],
}, name='value').to_frame())


,value
export_run_id,351a5af0-5bc1-50d0-ab20-fa8292afe787
export_content_hash,14f07a9f992758be294b41ff22e7ae5d9a4c771149ea6e...
model_config_hash,f40cba619da7e18be7b541bf12284101e28e39ddf8ff92...
selection_config_hash,037f61bfe3c1be8f7a1330e4b2f25306a32dced1f304a3...
seed,42


## Phase 4~5 실행

전처리 artifact cache가 있으면 동일 hash의 Parquet를 재사용합니다. 학습은 날짜 순서 walk-forward와 label-exit purge를 사용하며, 모델은 `SHADOW`로 등록됩니다.

In [5]:
from sp500_candidate_engine.analytics.phase45_workflow import run_phase45_workflow
from sp500_candidate_engine.config import Settings

summary = run_phase45_workflow(
    settings=Settings.from_env(REPO_ROOT / '.env'),
    export_dir=EXPORT_DIR,
    model_config=MODEL_CONFIG,
    selection_config=SELECTION_CONFIG,
    output_root=REPO_ROOT / 'data/output/phase45',
)
display(pd.Series({
    'experiment_key': summary.experiment_key,
    'champion_model': summary.champion_model,
    'model_version_id': str(summary.model_version_id),
    'scoring_run_id': str(summary.scoring_run_id),
    'selection_run_id': str(summary.selection_run_id),
    'candidate_count': summary.candidate_count,
    'price_session_date': summary.price_session_date,
}, name='value').to_frame())


AttributeError: RecorderWrapper(provider=None) does not have the attribute 'log_metrics'

## 평가 결과와 후보

점수는 확률이나 예상수익률이 아니라 ranking signal입니다. 후보는 투자 추천이 아니라 후속 분석 대상입니다.

In [ ]:
metrics = pd.DataFrame(summary.metrics).T
display(metrics.sort_values('rank_ic_mean', ascending=False))

candidate_payload = json.loads(summary.candidate_path.read_text(encoding='utf-8'))
display(pd.DataFrame(candidate_payload['candidates']))

assert 5 <= len(candidate_payload['candidates']) <= 10
assert candidate_payload['selection_type'] == 'QUANTITATIVE'
assert candidate_payload['price_session_date'] == str(summary.price_session_date)


## 재현성 확인

DB와 artifact에 기록된 ID·hash를 결과 manifest와 대조합니다. 재실행 시 동일 입력은 동일 idempotency identity를 사용합니다.

In [ ]:
metrics_manifest = json.loads((summary.experiment_dir / 'metrics.json').read_text(encoding='utf-8'))
assert metrics_manifest['experiment_key'] == summary.experiment_key
assert metrics_manifest['champion_model'] == summary.champion_model
print('reproducibility checks passed')
